# Data and downlink budget — how it works

This is the explanation and reference for the `quicksat` data budget. It covers the input files, and how the payload's data generation becomes a volume. It also shows how the downlink is compared with that volume, and where the tool stops.

It is not a tutorial, on purpose. In [Diátaxis](https://diataxis.fr/) terms, `sample/` holds the tutorials and how-to guides, and `docs/` holds the explanation and the reference. It still runs, because an explanation that cannot run soon stops matching the code it describes. It runs against `docs/sizing/data/`, its own copy of the sample satellite's input files. The copy is on purpose. The figures quoted below are true only of the data that produced them. If this notebook read `sample/sizing/data/`, a change to the tutorial's inputs would change the figures here without raising an error.

Every figure below comes from calling `quicksat.dataflow.budget`, rather than from repeating its arithmetic here. So the explanation and the module cannot disagree about what the chain computes.

In [1]:
import os
from pathlib import Path

import pandas as pd

# Make the in-development quicksat package importable without installing it: walk up
# from the current directory to the repo root (the folder that holds the quicksat
# package) and switch to it. Works whether the notebook runs from docs/, the repo
# root, or the docs build.
here = Path.cwd()
repo_root = next(
    (p for p in (here, *here.parents) if (p / "quicksat" / "__init__.py").exists()),
    None,
)
if repo_root is None:
    raise RuntimeError("could not locate the quicksat repo root")
os.chdir(repo_root)

from quicksat import Q_, u
from quicksat.dataflow.budget import DataBudget, DataFlowModel
from quicksat.utils.mission import Mission

pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

In [2]:
mission = Mission.from_yaml_file(Path("docs/sizing/data/mission.yaml"))
model = DataFlowModel.from_yaml_file(Path("docs/sizing/data/pl_dataflow_model.yaml"))
budget = DataBudget(model, mission)

generation, downlink = model.generation, model.downlink
budget

## The data model

The mass budget sums many rows. A data budget is instead **one chain of calculations**, from a few assumptions to a single comparison. The comparison asks whether the downlink can send everything that the payload generates. The inputs are not a list of things that exist. They are estimates of how the mission will behave. A duty cycle, a compression ratio and an average pass duration are all estimates, and none of them is a line item.

So the input is a config file rather than a CSV, and the output is a chain rather than a sum:

```
period          = 2*pi*sqrt(r^3/mu)              r = R_EARTH + altitude
orbits_per_day  = 1 day / period

gen_rate        = raw_datarate / compression_ratio
gen_duration    = duty_cycle * period
generated/orbit = gen_rate * gen_duration
generated/day   = generated/orbit * orbits_per_day

contact/day     = contacts_per_day * avg_contact_duration
downlink/day    = downlink_rate * contact/day
downlink/orbit  = downlink/day / orbits_per_day

margin          = downlink/day / generated/day - 1
storage         = generated/orbit * orbits_without_contact
```

Each line is a property of `DataBudget`, computed when it is read. There is no `resolve()` to call first, and no cache to clear. This is the other difference from the mass budget. With no rows to filter, there is no working table to build, only a few scalars computed from the inputs.

## The files

There are two files, split on purpose.

**`docs/sizing/data/mission.yaml`** is the shared mission, loaded into `quicksat.utils.mission.Mission`. This budget takes two things from it: the orbital period, and how many orbits fit in a day. It needs nothing else, and does not use the inclination.

**`mission_ref.ipynb`** explains how the mission is specified, validated and derived. The explanation is there rather than here, because the delta-V and agility budgets read the same file. So it belongs with the mission, rather than with one of the budgets that use it.

**`docs/sizing/data/pl_dataflow_model.yaml`** is the payload dataflow model. It holds generation, downlink and storage together. They stay in one file because the budget exists to compare the first two. Splitting the two sides of a comparison across files would make it harder to read, not easier.

| section | setting | meaning |
|---|---|---|
| `generation` | `raw_datarate` | instrument output *before* compression |
| | `compression_ratio` | divides the raw rate; 1 means no compression |
| | `duty_cycle` | fraction of each orbit spent generating |
| `downlink` | `rate` | achieved throughput |
| | `contacts_per_day` | usable ground contacts in a day, from Mission Analysis |
| | `avg_contact_duration` | how long an average contact lasts |
| `storage` | `orbits_without_contact` | the sizing case for onboard memory |

Contact statistics are **inputs**, not derived. Mission Analysis supplies how many contacts the mission gets each day, and how long they last.

In [3]:
print(Path("docs/sizing/data/pl_dataflow_model.yaml").read_text())

# Payload dataflow model: what the instrument generates, what the downlink clears,
# and what has to be held on board in between.
#
# Orbit comes from mission.yaml.

generation:
  raw_datarate: 1000 Mbit/s   # before compression; 416.67 Mbit/s after
  compression_ratio: 2.4
  duty_cycle: 5 %          # of each orbit: daytime, full VNIR

downlink:
  rate: 800 Mbit/s              # X-band transmitter throughput
  contacts_per_day: 7           # usable ground contacts, from Mission Analysis
  avg_contact_duration: 6 min   # average usable contact

storage:
  orbits_without_contact: 3   # sizing case for the mass memory



## Units, and what they rule out

Every rate and duration is text with a unit, parsed by [astropy](https://docs.astropy.org/en/stable/units/), like `unit_mass` in the equipment file. This gives a dimension check. A length where a data rate belongs is rejected, because it is not a rate at all.

astropy treats **the bit as a unit**. A data rate is bits over time, not a frequency. So two entries that look plausible are rejected on load:

- `800 MHz` is a frequency, not a data rate. A unit system that treats bits as dimensionless cannot tell the two apart.
- `800 Mbps` does not parse, because astropy has no `bps`. This is the safer failure. pint reads `bps` as baud: it computes the right number, then reports it as a *symbol* rate. A symbol can carry more than one bit, so that is the wrong quantity.

The sample files therefore write `Mbit/s`.

In [4]:
# a data rate is information over time, so a frequency is not one
for text in ("800 Mbit/s", "800 MHz", "800 Mbps"):
    try:
        q = Q_(text)
    except ValueError as exc:
        print(f"{text:12s} -> rejected: {str(exc).split(':')[0]}")
        continue
    verdict = "a data rate" if (u.bit / u.s).is_equivalent(q.unit) else "not a data rate"
    print(f"{text:12s} -> {q}   ({u.get_physical_type(q)}: {verdict})")

# a duty cycle is a dimensionless percentage, and astropy keeps that straight
print(f"\nduty cycle {generation.duty_cycle:g} -> "
      f"{generation.duty_cycle.to(u.dimensionless_unscaled):.3f} as a fraction")

800 Mbit/s   -> 800.0 Mbit / s   (bandwidth: a data rate)
800 MHz      -> 800.0 MHz   (frequency: not a data rate)
800 Mbps     -> rejected: 'Mbps' did not parse as unit

duty cycle 5 % -> 0.050 as a fraction


## The mission

Every daily figure in this budget is a figure per orbit, multiplied by the number of orbits in a day. So the budget takes only the period and the orbits per day from the mission:

$$T = 2\pi\sqrt{r^3/\mu}, \quad r = R_\oplus + h \qquad n_{\text{day}} = \frac{1\,\text{day}}{T}$$

`orbits_per_day` is dimensionless rather than a rate. So it can multiply a volume per orbit, and astropy raises no error about orbits not being seconds. `mission_ref.ipynb` covers the constants behind μ, the checks on load, and the limits of the circular orbit.

In [5]:
period = mission.period
orbits_per_day = mission.orbits_per_day

print(f"altitude        {mission.altitude:.1f}")
print(f"period          {period:.0f}  ({period.to('min'):.2f})")
print(f"orbits per day  {orbits_per_day:.3f}")

altitude        500.0 km
period          5677 s  (94.62 min)
orbits per day  15.219


## Generation

The payload generates at some rate for some fraction of each orbit.

**Compression is a separate input.** The file states the raw instrument data rate and the compression ratio separately, so the ratio can be varied. Giving only the compressed rate would hide the input that is most likely to be disputed.

**The duty cycle is a fraction of the orbit, not of the daylight pass.** 5% of a 5677 s orbit is 284 s of imaging.

In [6]:
print(f"raw rate          {generation.raw_datarate:.2f}")
print(f"compression       {generation.compression_ratio}x")
print(f"effective rate    {budget.effective_datarate:.2f}")
print(f"generation window {budget.generation_duration:.2f} per orbit")
print()
print(f"generated         {budget.generated_per_orbit:.2f} per orbit")
print(f"                  {budget.generated_per_day:.2f} per day")

raw rate          1000.00 Mbit / s
compression       2.4x
effective rate    416.67 Mbit / s
generation window 283.85 s per orbit

generated         14.78 Gbyte per orbit
                  225.00 Gbyte per day


## Downlink

Downlink has the same form as generation: a rate multiplied by a duration. What differs is the direction of the calculation.

Generation is counted **per orbit**, because the payload collects data for a fraction of each orbit. The daily figure is derived from it. Downlink is counted **per day**. Contacts are counted over the ground station's day, not over the spacecraft's orbit. The count does not divide evenly into orbits either. So each side of the comparison is computed over its own period, and the two are compared per day.

`contacts_per_day` and `avg_contact_duration` multiply to give the daily contact time. They are given separately rather than as one number, on purpose. A ground segment team uses this form. It also shows which of the two assumptions causes a change when the budget changes.

In [7]:
print(f"downlink rate     {downlink.rate:.0f}")
print(f"contacts          {downlink.contacts_per_day:g} per day at "
      f"{downlink.avg_contact_duration:.0f} each")
print(f"contact time      {budget.contact_per_day:.1f} per day")
print()
print(f"downlinked        {budget.downlinked_per_day:.2f} per day")
print(f"                  {budget.downlinked_per_orbit:.2f} per orbit "
      f"(derived, {orbits_per_day:.3f} orbits/day)")

downlink rate     800 Mbit / s
contacts          7 per day at 6 min each
contact time      42.0 min per day

downlinked        252.00 Gbyte per day
                  16.56 Gbyte per orbit (derived, 15.219 orbits/day)


## The margin, the main result

Everything above exists to produce the margin.

$$\text{margin} = \frac{\text{downlinked}}{\text{generated}} - 1$$

A positive margin means that each day's data is sent down the same day. A negative margin means data builds up until some is deleted or a pass is added. The margin is a ratio, so it is the one figure in this budget that **the byte convention cannot affect**. It is the same with GB or GiB, as long as both sides use the same one, because the convention cancels.

What the margin means for a particular mission is a question about that mission, not about the model. It covers how far each assumption can change before the margin is gone, and which input to change when it is. `sample/sizing/data_budget.ipynb` works this through for the sample satellite.

In [8]:
margin = budget.margin
balance = budget.downlinked_per_day - budget.generated_per_day

print(f"generated    {budget.generated_per_day:.2f} per day")
print(f"downlinked   {budget.downlinked_per_day:.2f} per day")
print(f"margin       {margin.to('percent'):+.2f}")
print(f"{'surplus' if balance > 0 else 'shortfall':12s} {abs(balance):.2f} per day")

generated    225.00 Gbyte per day
downlinked   252.00 Gbyte per day
margin       +12.00 %
surplus      27.00 Gbyte per day


## Bytes: decimal, not binary

quicksat uses **decimal** bytes throughout, so `GB` is 10⁹ bytes. Link rates are quoted in decimal units, so this keeps one convention across the whole chain. astropy enforces it, rather than a comment.

This needs saying, because a volume in bytes is ambiguous unless the convention is named. A common habit is to divide megabits by 8, and then by 1024. This mixes the two conventions: decimal megabits become binary gigabytes. All three readings describe the same quantity, and differ by up to about 2.3%, as the table below shows. Someone who compares two figures by hand will find that gap, and may suspect a bug. It comes from the convention, not from an error.

The figures that do not depend on the convention are unaffected: contact minutes, contact count, and the margin.

In [9]:
raw_bits = budget.effective_datarate * budget.generation_duration
comparison = pd.DataFrame(
    {
        "per orbit": [
            raw_bits.to("GB").value,
            raw_bits.to("GiB").value,
            raw_bits.to("Mbit").value / 8 / 1024,
        ]
    },
    index=["decimal GB (quicksat)", "binary GiB", "mixed (Mbit / 8 / 1024)"],
)
comparison["vs quicksat"] = (
    comparison["per orbit"] / comparison.loc["decimal GB (quicksat)", "per orbit"] - 1
) * 100
comparison

,per orbit,vs quicksat
decimal GB (quicksat),14.784,0.000
binary GiB,13.768,-6.868
mixed (Mbit / 8 / 1024),14.437,-2.344


## Storage

The onboard memory has to hold the data that builds up between contacts. quicksat sizes it the simple way: the data generated over a given number of orbits without a pass. It does not simulate a growing backlog over the mission.

This model is rough on purpose, and that suits a sizing tool. The input is the number of orbits that might pass without a usable contact. A ground segment team can give that number. The alternative needs a contact schedule, and nobody has one yet.

This figure does **not** cover a growing backlog. It sizes the memory for the gap between passes. The difference matters, because the two are the same only while the budget closes. With a negative margin, the backlog would never clear, and the storage needed would grow without limit. This number would then be meaningless, not only approximate.

In [10]:
print(f"generated per orbit      {budget.generated_per_orbit:.2f}")
print(f"orbits without contact   {model.storage.orbits_without_contact:g}")
print(f"storage required         {budget.storage_required():.2f}")
print(f"  overridable:           {budget.storage_required(1):.2f} for a single orbit")

# what the mass budget says is actually flying
equipment = pd.read_csv("docs/sizing/data/equipment.csv")
ssdr = equipment.query("equipment_id == 'mass_memory'").iloc[0]
print(f"\nflying, per the equipment list: {ssdr['equipment_name']} ({ssdr['unit_mass']})")
print(f"  required {budget.storage_required().to('TB'):.3f} against a 2 TB device - ample,")
print("  before any allowance for file system overhead, redundancy or degradation")

generated per orbit      14.78 Gbyte
orbits without contact   3
storage required         44.35 Gbyte
  overridable:           14.78 Gbyte for a single orbit

flying, per the equipment list: SSDR 2 TB (4.5 kg)
  required 0.044 Tbyte against a 2 TB device - ample,
  before any allowance for file system overhead, redundancy or degradation


## The class

`DataFlowModel` and `Mission` are the two inputs, and `DataBudget` is the chain of calculations over them. Both models validate on load. So a malformed file raises an error that names the bad field, rather than failing later in the arithmetic.

There are two ways to build the budget. The choice depends on whether anything else needs the mission:

- **`DataBudget.from_yaml_file(model_path, mission_path)`** reads both files. It builds one budget in one line, with nothing else to keep consistent.
- **`DataBudget(model, mission)`** takes objects that are already loaded. Use it when the delta-V budget is in the same notebook. The mission is then read once, and every budget visibly uses the same one. Otherwise three budgets would read three copies that only happen to agree.

Everything after that is a property, named for what it is *per*: `effective_datarate`, `generation_duration`, `generated_per_orbit`, `generated_per_day`, `contact_per_day`, `downlinked_per_day`, `downlinked_per_orbit`, and `margin`. `storage_required()` is the only method, and only because it takes an override.

There is no setter anywhere. To try a different case, build another `DataFlowModel`. This takes one line from a dict. It also keeps the new case separate from the case in the file.

### The document view

`tabulated_data()` lays out the whole chain as a document. Each row gives the item, its value, its unit, and the assumption behind it, in the order the chain computes them. It returns a pandas `Styler`, so the figures print at a fixed number of decimals. Two rows hold results rather than steps: the margin and the storage. They are printed in bold.

The frame is still there as `.data`, with the one column that the display hides. That column tags every row. `input` marks a value typed into a file, and `derived` a value that the chain worked out from the inputs. `margin` and `storage` mark the two results. The tag makes the report filterable. The inputs of one case can be selected and compared with another's, without parsing captions.

In [11]:
budget.tabulated_data()

,Value,Unit,Comment
Orbital period,94.62,min,500 km circular
Orbits per day,15.22,-,
Data generation rate,416.67,Mbit/s,"1000 Mbit / s raw, 2.4x compression"
Data generation duration,283.85,s/orbit,5 % duty cycle
Data generated,14.78,GB/orbit,
,225.00,GB/day,
Data downlink rate,800.00,Mbit/s,achieved throughput
Contacts,7.00,per day,6 min average duration
Data downlink duration,42.00,min/day,
Data downlinked,16.56,GB/orbit,


In [12]:
report = budget.tabulated_data().data
print(report["row_type"].value_counts().to_dict())

# what was typed into a file, as opposed to what the chain worked out
print()
print(report.query("row_type == 'input'")[["item", "value", "unit"]].to_string(index=False))

{'derived': 6, 'input': 5, 'margin': 1, 'storage': 1}

                    item   value    unit
          Orbital period  94.616     min
    Data generation rate 416.667  Mbit/s
Data generation duration 283.847 s/orbit
      Data downlink rate 800.000  Mbit/s
                Contacts   7.000 per day


## Limitations

What the data budget does not do, on purpose, roughly in order of how often it comes up:

- **No contact schedule.** Passes are an average count and an average duration. There is no station list, no elevation mask and no propagator. So the budget knows nothing of a gap at a particular time of day. In a real design, that gap is what sizes the storage.
- **Storage is a multiple of one orbit's generation.** There is no growing backlog and no search for the worst gap. There is no margin for a missed pass beyond the number you enter.
- **One payload, one mode.** There is a single rate and duty cycle. An instrument with a high-rate mode and a survey mode, or a second payload, has to be combined into one equivalent rate by hand.
- **Rates are achieved throughput.** Nothing here models modulation, coding, link margin, rain fade or protocol overhead. `800 Mbit/s` is a statement of what the link delivers. It would come from a link budget, which quicksat does not have.
- **No latency.** The budget says whether the whole volume is sent down. It never says how old the oldest unsent frame is. That is often the requirement that limits the design.
- **Compression is a single ratio.** It is constant across the scene and independent of content. There is no distinction between lossy and lossless.
- **Circular orbit.** The period comes from the altitude. There is no eccentricity, and no decay from drag over the mission.